# v4 Phase 0 — Diagnostics on the v3 models (read-only)

**Why this notebook exists.** The 2026-10-12 discussion (`sessions/2026-10-12.md`, Sections 9–11)
raised four questions that decide what v4 should change. Each is answered from the models v3
already trained. Nothing is trained, and no v3 file is written.

| # | Question | Why it matters |
|---|---|---|
| D1 | Are the collisions drone–drone or drone–obstacle? | Tells us whether obstacle sensing is the weak point |
| D2 | During training, how often was the safety reward switched on? | Hypothesis: PAH has little to arbitrate because danger is rare |
| D3 | How much does α actually move? | Is PAH effectively a fixed α most of the time? |
| D4 | Loss-of-separation and active conflict edges (IGAT-MARL's metrics) | Base-paper metric; shows near-misses that success rate hides |

**Two passes.**
- *Evaluation pass* (D1, D4): every final v3 model (18), deterministic policy, 200 episodes, env seed 7001 —
  the same protocol and episode layouts as `v3/results-table`.
- *Training-conditions pass* (D2, D3): every saved checkpoint (~every 500 episodes) of the seed-42 runs,
  **stochastic** policy (sampled actions, as in the training rollout), 50 episodes each, env seed 9101.

**Definitions.** Danger radius = 3 × collision radius = **9 units** — the radius the v3 reward uses for
`r_safety` and the conflict graph uses for an edge. Loss-of-separation (LoS) = a drone pair closer than 9
units at a step (drone–drone only, as in IGAT-MARL). Active edges = conflict-graph edges (pairs on a
collision course within the 3-step look-ahead). α enters training by weighting *advantages*
(`α·A_mission + (1−α)·A_safety`); because GAE carries the safety advantage a few steps back from a danger
step, "safety active %" is a lower bound on how often α can matter, not an exact count.

**Code.** Environment + networks: `../common/v3_env.py` (one frozen copy, verified episode-by-episode
against v3). Model paths: `../common/v3_registry.py`. Instrumented rollout: `diag.py`.

In [1]:
# Cell 1 -- setup
import os, sys, json, time, math
import numpy as np
import pandas as pd
import torch
torch.set_num_threads(1)

sys.path.insert(0, os.path.abspath('../common'))
import v3_registry as R
import diag as D

OUT = 'output'
os.makedirs(OUT, exist_ok=True)

N_EVAL, EVAL_SEED = 200, 7001     # same as v3/results-table
N_TRAIN, TRAIN_SEED = 50, 9101    # training-conditions pass (new, disjoint seed)
TRAIN_RUNS = [(2, 'B4'), (2, 'H'), (2, 'M'), (3, 'B4'), (3, 'M'), (3, 'M-noHungarian')]   # seed 42

print(R.check_all())

18 models, all final_model.pt found


In [2]:
# Cell 2 -- GATE: the frozen copy must reproduce v3 episode-by-episode, or stop here
RAW_V3 = os.path.join(R.V3, 'results-table', 'output', 'raw')

def outcomes(entry, n=N_EVAL, seed=EVAL_SEED):
    actor, pah = D.load_checkpoint(R.final_model(entry))
    env = D.make_env(entry, seed)
    return [D.run_episode(env, actor, pah, entry['method'], 'det')[0]['outcome'] for _ in range(n)]

for stage, method, seed in [(3, 'M', 42), (2, 'B4', 43)]:
    e = R.find(stage, method, seed)
    mine = outcomes(e)
    ref = [r['outcome'] for r in json.load(open('{}/s{}_{}_seed{}.json'.format(RAW_V3, stage, method, seed)))['records']]
    same = sum(a == b for a, b in zip(mine, ref))
    print('Stage {} {} seed {}: {}/{} episodes identical to v3'.format(stage, method, seed, same, len(ref)))
    assert same == len(ref), 'GATE FAILED -- do not trust anything below'
print('GATE PASSED')

Stage 3 M seed 42: 200/200 episodes identical to v3


Stage 2 B4 seed 43: 200/200 episodes identical to v3
GATE PASSED


In [3]:
# Cell 3 -- Evaluation pass (D1, D4): all 18 final models, deterministic, 200 episodes
EVAL_CSV = OUT + '/eval_episodes.csv'
if os.path.exists(EVAL_CSV):
    ev = pd.read_csv(EVAL_CSV, keep_default_na=False)
    print('loaded cached', EVAL_CSV)
else:
    rows = []
    for e in R.REGISTRY:
        t0 = time.time()
        actor, pah = D.load_checkpoint(R.final_model(e))
        env = D.make_env(e, EVAL_SEED)
        for k in range(N_EVAL):
            s, _ = D.run_episode(env, actor, pah, e['method'], 'det')
            if s['outcome'] == 'collision':
                # independent check: the env's own collision test must agree with the kind we report
                assert env._any_collision() and s['collision_kind'] != 'unknown'
            rows.append(dict(stage=e['stage'], method=e['method'], seed=e['seed'], episode=k, **s))
        print('stage {} {:14s} seed {}  [{:.0f}s]'.format(e['stage'], e['method'], e['seed'], time.time() - t0))
    ev = pd.DataFrame(rows)
    ev.to_csv(EVAL_CSV, index=False)
print(len(ev), 'episodes')

loaded cached output/eval_episodes.csv
3600 episodes


In [4]:
# Cell 4 -- Training-conditions pass (D2, D3): every checkpoint, stochastic, 50 episodes
TRAIN_CSV = OUT + '/training_conditions.csv'
if os.path.exists(TRAIN_CSV):
    tr = pd.read_csv(TRAIN_CSV)
    print('loaded cached', TRAIN_CSV)
else:
    rows = []
    for stage, method in TRAIN_RUNS:
        e = R.find(stage, method, 42)
        ckpts = R.checkpoints(e) + [(8000, R.final_model(e))]   # final model plotted at episode 8000
        t0 = time.time()
        for ep, path in ckpts:
            actor, pah = D.load_checkpoint(path)
            env = D.make_env(e, TRAIN_SEED)
            gen = torch.Generator().manual_seed(TRAIN_SEED)
            S, A = [], []
            for _ in range(N_TRAIN):
                s, a = D.run_episode(env, actor, pah, method, 'stoch', gen)
                S.append(s); A.append(a)
            S = pd.DataFrame(S); A = np.concatenate(A)
            # drone-step weighted averages (longer episodes count more, like a training batch)
            w = S['steps'] * e['n_drones']
            avg = lambda col: float(np.average(S[col], weights=w))
            rows.append(dict(
                stage=stage, method=method, episode=ep, final=(path == R.final_model(e)),
                success=100 * (S.outcome == 'success').mean(), collision=100 * (S.outcome == 'collision').mean(),
                safety_pct=100 * avg('safety_frac'), safety_drone_pct=100 * avg('safety_drone_frac'),
                safety_obs_pct=100 * avg('safety_obs_frac'), conflict_pct=100 * avg('conflict_frac'),
                alpha_mean=float(A.mean()), alpha_p10=float(np.percentile(A, 10)), alpha_p50=float(np.percentile(A, 50)),
                alpha_p90=float(np.percentile(A, 90)), alpha_lt_05_pct=100 * float((A < 0.5).mean()),
                alpha_lt_08_pct=100 * float((A < 0.8).mean()), drone_steps=int(w.sum())))
        print('stage {} {:14s}: {} checkpoints [{:.0f}s]'.format(stage, method, len(ckpts), time.time() - t0))
    tr = pd.DataFrame(rows)
    tr.to_csv(TRAIN_CSV, index=False)
print(len(tr), 'checkpoint rows')

loaded cached output/training_conditions.csv
95 checkpoint rows


In [5]:
# Cell 5 -- Tables
pd.set_option('display.width', 160)
LABEL = {'B4': 'B4 (fixed α)', 'H': 'H (τ-formula)', 'M': 'M (PAH)', 'M-noHungarian': 'M, no Hungarian'}
ORDER = {2: ['B4', 'H', 'M'], 3: ['B4', 'M', 'M-noHungarian']}

# D1 -- what was hit (all collision episodes, seeds pooled)
d1 = []
for st in (2, 3):
    for m in ORDER[st]:
        c = ev[(ev.stage == st) & (ev.method == m) & (ev.outcome == 'collision')]
        n = len(c)
        d1.append(dict(stage=st, method=LABEL[m], collisions=n,
                       drone_obstacle=int((c.collision_kind == 'drone-obstacle').sum()),
                       drone_drone=int((c.collision_kind == 'drone-drone').sum()),
                       both=int((c.collision_kind == 'both').sum()),
                       obstacle_pct=round(100 * (c.collision_kind != 'drone-drone').mean(), 1) if n else float('nan'),
                       near_goal_pct=round(100 * c.collision_near_goal.astype(bool).mean(), 1) if n else float('nan')))
D1 = pd.DataFrame(d1); print('D1 -- collision kinds (evaluation pass)'); print(D1.to_string(index=False)); print()

# D4 -- IGAT-MARL metrics per method (Stage 2: mean +/- std over seed means; Stage 3: mean +/- 95% CI over episodes)
d4 = []
for st in (2, 3):
    for m in ORDER[st]:
        x = ev[(ev.stage == st) & (ev.method == m)]
        if st == 2:
            g = x.groupby('seed')[['los_pair_steps', 'los_rate', 'edges_mean']].mean()
            mean, err = g.mean(), g.std(ddof=1)
            note = 'mean ± std over 5 seeds'
        else:
            mean = x[['los_pair_steps', 'los_rate', 'edges_mean']].mean()
            err = 1.96 * x[['los_pair_steps', 'los_rate', 'edges_mean']].std(ddof=1) / math.sqrt(len(x))
            note = 'mean ± 95% CI over 200 episodes'
        d4.append(dict(stage=st, method=LABEL[m], los_per_episode=mean['los_pair_steps'], los_err=err['los_pair_steps'],
                       los_per_step=mean['los_rate'], edges_per_step=mean['edges_mean'], edges_err=err['edges_mean'], note=note))
D4 = pd.DataFrame(d4); print('D4 -- loss-of-separation and active edges (evaluation pass)'); print(D4.round(3).to_string(index=False)); print()

# D2 / D3 -- training conditions, early (first checkpoint) / middle / final
def pick(df):
    df = df.sort_values('episode')
    return pd.concat([df.iloc[[0]], df.iloc[[len(df) // 2]], df.iloc[[-1]]])
cols = ['stage', 'method', 'episode', 'success', 'safety_pct', 'safety_drone_pct', 'safety_obs_pct',
        'conflict_pct', 'alpha_p10', 'alpha_p50', 'alpha_p90', 'alpha_lt_05_pct', 'alpha_lt_08_pct']
D23 = pd.concat([pick(tr[(tr.stage == st) & (tr.method == m)]) for st, m in TRAIN_RUNS])[cols]
print('D2/D3 -- training conditions (stochastic), first / middle / final checkpoint')
print(D23.round(2).to_string(index=False))

with open(OUT + '/tables.md', 'w') as f:
    f.write('## D1 -- collision kinds (evaluation pass)\n\n' + D1.to_markdown(index=False) + '\n\n')
    f.write('## D4 -- loss-of-separation and active edges (evaluation pass)\n\n' + D4.round(3).to_markdown(index=False) + '\n\n')
    f.write('## D2/D3 -- training conditions (stochastic)\n\n' + D23.round(2).to_markdown(index=False) + '\n')
print('\nsaved', OUT + '/tables.md')

D1 -- collision kinds (evaluation pass)
 stage          method  collisions  drone_obstacle  drone_drone  both  obstacle_pct  near_goal_pct
     2    B4 (fixed α)         108             107            1     0          99.1           20.4
     2   H (τ-formula)         125             124            1     0          99.2           20.8
     2         M (PAH)         105             104            1     0          99.0           21.9
     3    B4 (fixed α)          53              50            3     0          94.3           30.2
     3         M (PAH)          56              56            0     0         100.0           39.3
     3 M, no Hungarian          97              78           19     0          80.4           19.6

D4 -- loss-of-separation and active edges (evaluation pass)
 stage          method  los_per_episode  los_err  los_per_step  edges_per_step  edges_err                            note
     2    B4 (fixed α)            0.098    0.038         0.002           0.002      

In [6]:
# Cell 6 -- Figures (white background, same house style as v3/results-table)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white', 'savefig.facecolor': 'white',
    'font.family': 'DejaVu Sans', 'font.size': 9.5, 'axes.titlesize': 10.5, 'axes.labelsize': 9.5,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'axes.axisbelow': True,
    'grid.alpha': 0.25})
# method colours = the ones v3 fig6 already uses, so the thesis stays consistent
MCOL = {'B4': '#808080', 'H': '#D98E2B', 'M': '#3B6FB6', 'M-noHungarian': '#C0392B'}
# collision-kind colours: validated categorical slots (blue / orange / aqua)
KCOL = {'drone-obstacle': '#2a78d6', 'drone-drone': '#eb6834', 'both': '#1baf7a'}

def save(fig, name):
    fig.savefig('{}/{}.png'.format(OUT, name), dpi=200, bbox_inches='tight')
    fig.savefig('{}/{}.svg'.format(OUT, name), bbox_inches='tight')
    plt.close(fig)

# ---- D1: what was hit --------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, st in zip(axes, (2, 3)):
    sub = D1[D1.stage == st]
    x = np.arange(len(sub)); bottom = np.zeros(len(sub))
    for kind, col in (('drone_obstacle', 'drone-obstacle'), ('drone_drone', 'drone-drone'), ('both', 'both')):
        v = sub[kind].values
        ax.bar(x, v, 0.55, bottom=bottom, color=KCOL[col], edgecolor='white', linewidth=2, label=col)
        bottom += v
    for xi, tot, pct in zip(x, sub.collisions.values, sub.obstacle_pct.values):
        ax.text(xi, tot + bottom.max() * 0.02, '{} collisions\n{:.0f}% hit an obstacle'.format(tot, pct),
                ha='center', va='bottom', fontsize=8.3)
    ax.set_xticks(x); ax.set_xticklabels(sub.method)
    ax.set_ylim(0, bottom.max() * 1.3); ax.set_ylabel('Collision episodes (of 200 per model)')
    ax.set_title('Stage {} — {}'.format(st, '5 seeds pooled (1000 episodes per method)' if st == 2 else 'seed 42 (200 episodes)'))
axes[0].legend(loc='upper left', frameon=False, fontsize=8.5)
fig.suptitle('D1 — What did the drones hit? (evaluation pass: deterministic policy)', fontsize=10.5)
plt.tight_layout(rect=(0, 0, 1, 0.94)); save(fig, 'd1_collision_kinds')

# ---- D2: how often was the safety reward on, during training -------------------
fig, axes = plt.subplots(2, 2, figsize=(11.5, 7.2), sharex=True)
for col, st in enumerate((2, 3)):
    for (s2, m) in TRAIN_RUNS:
        if s2 != st: continue
        d = tr[(tr.stage == st) & (tr.method == m)].sort_values('episode')
        axes[0, col].plot(d.episode, d.safety_pct, color=MCOL[m], lw=2, marker='o', ms=3.5, label=LABEL[m])
        axes[1, col].plot(d.episode, d.conflict_pct, color=MCOL[m], lw=2, marker='o', ms=3.5, label=LABEL[m])
    axes[0, col].set_title('Stage {} (seed 42)'.format(st))
    axes[1, col].set_xlabel('Training episode of the checkpoint')
axes[0, 0].set_ylabel('% of drone-steps with\nsafety reward ON (r_safety ≠ 0)')
axes[1, 0].set_ylabel('% of drone-steps the conflict\ngraph flags (PAH sees a conflict)')
for ax in axes.ravel():
    ax.set_ylim(bottom=0)
axes[0, 0].legend(frameon=False, fontsize=8.5)
axes[0, 1].legend(frameon=False, fontsize=8.5)
fig.suptitle('D2 — How often is there anything to arbitrate? (training conditions: stochastic policy, 50 episodes per checkpoint)', fontsize=10.5)
plt.tight_layout(rect=(0, 0, 1, 0.95)); save(fig, 'd2_safety_active')

# ---- D3: how much does alpha move, during training ------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3), sharey=True)
for ax, st in zip(axes, (2, 3)):
    for (s2, m) in TRAIN_RUNS:
        if s2 != st or m == 'B4': continue
        d = tr[(tr.stage == st) & (tr.method == m)].sort_values('episode')
        ax.fill_between(d.episode, d.alpha_p10, d.alpha_p90, color=MCOL[m], alpha=0.18, lw=0)
        ax.plot(d.episode, d.alpha_p50, color=MCOL[m], lw=2, marker='o', ms=3.5, label=LABEL[m] + ' — median, band 10–90%')
    ax.axhline(0.5, color=MCOL['B4'], lw=1.6, ls='--', label='B4: fixed α = 0.5')
    ax.set_ylim(0, 1); ax.set_xlabel('Training episode of the checkpoint'); ax.set_title('Stage {} (seed 42)'.format(st))
    ax.legend(frameon=False, fontsize=8, loc='lower right')
axes[0].set_ylabel('α over all drone-steps')
fig.suptitle('D3 — How much does α actually move? (training conditions: stochastic policy)', fontsize=10.5)
plt.tight_layout(rect=(0, 0, 1, 0.93)); save(fig, 'd3_alpha_spread')

# ---- D4: IGAT-MARL metrics --------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3))
for ax, metric, err, ylab, fmt in ((axes[0], 'los_per_episode', 'los_err', 'Loss-of-separation pair-steps per episode', '{:.2f}'),
                                   (axes[1], 'edges_per_step', 'edges_err', 'Active conflict edges per step', '{:.3f}')):
    labels, vals, errs, cols = [], [], [], []
    for st in (2, 3):
        for m in ORDER[st]:
            r = D4[(D4.stage == st) & (D4.method == LABEL[m])].iloc[0]
            labels.append('S{}\n{}'.format(st, LABEL[m])); vals.append(r[metric]); errs.append(r[err]); cols.append(MCOL[m])
    x = np.arange(len(vals))
    ax.bar(x, vals, 0.6, yerr=errs, color=cols, edgecolor='white', linewidth=2, capsize=3, error_kw=dict(lw=1))
    for xi, v, e in zip(x, vals, errs):
        ax.text(xi, v + e + max(vals) * 0.02, fmt.format(v), ha='center', va='bottom', fontsize=8.3)
    ax.axvline(2.5, color='#bbb', lw=1, ls=':')
    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8.2); ax.set_ylabel(ylab)
    ax.set_ylim(0, max(v + e for v, e in zip(vals, errs)) * 1.25)
fig.suptitle('D4 — IGAT-MARL metrics on our models (evaluation pass). Stage 2: mean ± std over 5 seeds; '
             'Stage 3: mean ± 95% CI over 200 episodes', fontsize=10)
plt.tight_layout(rect=(0, 0, 1, 0.93)); save(fig, 'd4_los_edges')
print('saved d1_collision_kinds, d2_safety_active, d3_alpha_spread, d4_los_edges')

saved d1_collision_kinds, d2_safety_active, d3_alpha_spread, d4_los_edges


In [7]:
# Cell 7 -- Headline numbers (written to summary.json; README quotes these)
def row(st, m, final=True):
    d = tr[(tr.stage == st) & (tr.method == m)].sort_values('episode')
    return d.iloc[-1] if final else d
summary = {
    'D1_collision_kinds': D1.to_dict(orient='records'),
    'D4_igat_metrics': D4.round(4).to_dict(orient='records'),
    'D2_safety_on_pct_over_training': {
        '{}-S{}'.format(m, st): dict(min=round(float(row(st, m, False).safety_pct.min()), 2),
                                    max=round(float(row(st, m, False).safety_pct.max()), 2),
                                    final=round(float(row(st, m).safety_pct), 2),
                                    obstacle_part_final=round(float(row(st, m).safety_obs_pct), 2),
                                    drone_part_final=round(float(row(st, m).safety_drone_pct), 2))
        for st, m in TRAIN_RUNS},
    'D3_alpha_final': {
        '{}-S{}'.format(m, st): dict(p10=round(float(row(st, m).alpha_p10), 3), p50=round(float(row(st, m).alpha_p50), 3),
                                    p90=round(float(row(st, m).alpha_p90), 3),
                                    lt_05_pct=round(float(row(st, m).alpha_lt_05_pct), 2),
                                    lt_08_pct=round(float(row(st, m).alpha_lt_08_pct), 2))
        for st, m in TRAIN_RUNS if m != 'B4'},
}
json.dump(summary, open(OUT + '/summary.json', 'w'), indent=1, default=str)
print(json.dumps({k: summary[k] for k in ('D2_safety_on_pct_over_training', 'D3_alpha_final')}, indent=1))

{
 "D2_safety_on_pct_over_training": {
  "B4-S2": {
   "min": 0.19,
   "max": 0.51,
   "final": 0.49,
   "obstacle_part_final": 0.49,
   "drone_part_final": 0.0
  },
  "H-S2": {
   "min": 0.24,
   "max": 0.48,
   "final": 0.44,
   "obstacle_part_final": 0.41,
   "drone_part_final": 0.03
  },
  "M-S2": {
   "min": 0.23,
   "max": 0.62,
   "final": 0.3,
   "obstacle_part_final": 0.3,
   "drone_part_final": 0.0
  },
  "B4-S3": {
   "min": 0.81,
   "max": 1.31,
   "final": 1.27,
   "obstacle_part_final": 0.86,
   "drone_part_final": 0.41
  },
  "M-S3": {
   "min": 0.85,
   "max": 1.26,
   "final": 1.07,
   "obstacle_part_final": 0.97,
   "drone_part_final": 0.1
  },
  "M-noHungarian-S3": {
   "min": 1.2,
   "max": 1.71,
   "final": 1.35,
   "obstacle_part_final": 0.74,
   "drone_part_final": 0.61
  }
 },
 "D3_alpha_final": {
  "H-S2": {
   "p10": 0.9,
   "p50": 0.9,
   "p90": 0.9,
   "lt_05_pct": 0.0,
   "lt_08_pct": 0.02
  },
  "M-S2": {
   "p10": 0.742,
   "p50": 0.892,
   "p90": 0.9,
  